# Agente de priorizacion de inspecciones

**Interfaz conversacional sobre el sistema de deteccion de fallas del APS**

---

Los dos cuadernos anteriores producen un sistema que asigna a cada camion un
puntaje de riesgo y decide si conviene inspeccionarlo. Ese resultado vive en un
archivo y en un modelo serializado: util para un analista, inservible para un
jefe de taller a las seis de la manana.

Este cuaderno construye la capa que falta. Un agente sobre la API de Claude que
atiende preguntas en lenguaje natural —cuantos camiones revisar hoy, por que ese
en concreto, a cuales priorizar si solo hay capacidad para doscientos— y las
responde consultando el sistema.

## El principio de diseno

**El agente no decide nada y no calcula nada.** La decision la produce el modelo
entrenado; las cifras las producen funciones de Python. El agente traduce entre
el lenguaje del taller y esas dos cosas.

Esta separacion no es una restriccion tecnica sino el requisito central. Un
asistente que estime un puntaje de memoria, redondee un costo o suponga el
identificador que el usuario quiso escribir es peor que no tener asistente: sus
errores son indistinguibles de sus aciertos. Toda cifra que el agente comunica
proviene de una llamada verificable, y el cuaderno incluye una comprobacion
explicita de que asi ocurre.

## 1. Preparacion

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import json
import herramientas
from agente import Agente

print("Herramientas disponibles:")
for e in herramientas.ESQUEMAS:
    print(f"  {e['name']}")

Herramientas disponibles:
  resumen_flota
  consultar_camion
  priorizar_inspecciones
  generar_orden_trabajo
  explicar_sistema
  evaluar_desempeno


## 2. Las herramientas

Seis funciones. Cada una responde una clase de pregunta del taller y devuelve
datos estructurados, nunca texto redactado: la redaccion es trabajo del agente,
el dato es trabajo de la herramienta.

| Herramienta | Pregunta que responde |
|---|---|
| `resumen_flota` | Cual es la situacion general |
| `consultar_camion` | Que pasa con este camion |
| `priorizar_inspecciones` | A cuales atiendo si no me alcanza |
| `generar_orden_trabajo` | Que le entrego al mecanico |
| `explicar_sistema` | Como funciona esto y que no puede hacer |
| `evaluar_desempeno` | Que tan bien funciono |

### 2.1 Una decision de diseno: separar operacion de evaluacion

La flota puntuada conserva el desenlace real de cada camion, porque el conjunto
de prueba lo incluye. Esa informacion es imprescindible para evaluar el sistema
y **no debe estar disponible al decidir**: en operacion, cuando hay que resolver
si se envia un mecanico, todavia no se sabe si el camion iba a averiarse.

Por eso ninguna herramienta de uso diario expone ese dato. Solo lo hace
`evaluar_desempeno`, que responde preguntas retrospectivas y advierte de ello en
su propia salida. Si las herramientas de operacion lo devolvieran, el agente
acabaria respondiendo "este camion si tiene una averia" en lugar de "el sistema
recomienda revisarlo", que es una afirmacion distinta y que en produccion seria
imposible.

In [2]:
# Las herramientas funcionan por si solas, sin intervencion del modelo de lenguaje
print(json.dumps(herramientas.resumen_flota(), indent=2, ensure_ascii=False))

{
  "camiones_evaluados": 16000,
  "camiones_senalados": 745,
  "porcentaje_de_la_flota": 4.7,
  "umbral_de_riesgo": 0.008,
  "costo_politica_recomendada": 10840,
  "costo_no_inspeccionar_a_nadie": 187500,
  "costo_inspeccionar_a_todos": 156250,
  "reduccion_vs_correctivo_pct": 94.2,
  "unidades": "Los costos estan en las unidades definidas por la operacion: 10 por inspeccion innecesaria, 500 por averia no detectada."
}


In [3]:
# Consulta de un camion concreto
primero = herramientas.priorizar_inspecciones(1)["camiones"][0]["camion_id"]
ejemplo = herramientas.consultar_camion(primero)
print(json.dumps(ejemplo, indent=2, ensure_ascii=False))

{
  "encontrado": true,
  "camion_id": "T15298",
  "puntaje_de_riesgo": 0.999999,
  "umbral_de_inspeccion": 0.008,
  "decision": "INSPECCIONAR",
  "veces_el_umbral": 125.0,
  "posicion_en_la_flota": 1,
  "total_camiones": 16000,
  "sensores_que_motivan": {
    "hacia_inspeccionar": [
      {
        "sensor": "ag",
        "contribucion": 2.7996
      },
      {
        "sensor": "ay",
        "contribucion": 1.002
      },
      {
        "sensor": "bs",
        "contribucion": 0.8057
      },
      {
        "sensor": "ck",
        "contribucion": 0.804
      },
      {
        "sensor": "bd",
        "contribucion": 0.7997
      }
    ],
    "en_contra_de_inspeccionar": [
      {
        "sensor": "cb",
        "contribucion": -0.251
      },
      {
        "sensor": "du",
        "contribucion": -0.2006
      },
      {
        "sensor": "ec",
        "contribucion": -0.1745
      }
    ],
    "nota": "Solo las lecturas de 'hacia_inspeccionar' justifican el aviso. Las otras empuja

### 2.2 Los errores se devuelven, no se lanzan

Una excepcion interrumpe la conversacion; un error descrito permite al agente
explicar que paso. Todas las llamadas pasan por `ejecutar`, que captura
cualquier fallo y lo convierte en una respuesta utilizable.

In [4]:
print("Camion inexistente:")
print(json.dumps(herramientas.ejecutar("consultar_camion", {"camion_id": "T99999"}),
                 indent=2, ensure_ascii=False))
print()
print("Argumento equivocado:")
print(json.dumps(herramientas.ejecutar("consultar_camion", {"placa": "ABC-123"}),
                 indent=2, ensure_ascii=False))
print()
print("Herramienta que no existe:")
print(json.dumps(herramientas.ejecutar("borrar_flota", {}),
                 indent=2, ensure_ascii=False))

Camion inexistente:
{
  "encontrado": false,
  "camion_id": "T99999",
  "mensaje": "No hay ningun camion con identificador T99999 en la flota evaluada. Los identificadores van de T00000 a T15999."
}

Argumento equivocado:
{
  "error": "argumentos_invalidos",
  "detalle": "consultar_camion() got an unexpected keyword argument 'placa'"
}

Herramienta que no existe:
{
  "error": "Herramienta desconocida: borrar_flota",
  "disponibles": [
    "consultar_camion",
    "evaluar_desempeno",
    "explicar_sistema",
    "generar_orden_trabajo",
    "priorizar_inspecciones",
    "resumen_flota"
  ]
}


## 3. El agente

El ciclo es el habitual de uso de herramientas: se envia la conversacion junto
con la declaracion de las seis funciones, y mientras el modelo pida ejecutar
alguna se ejecuta y se le devuelve el resultado. Cuando deja de pedir
herramientas, su texto es la respuesta.

Dos detalles del bucle que importan en produccion: hay una cota al numero de
vueltas, para que un ciclo de peticiones no se prolongue indefinidamente, y el
agente registra cada llamada realizada, lo que permite auditar despues de donde
salio cada cifra.

Las instrucciones del sistema fijan las reglas de conducta. La central: ninguna
cifra sin herramienta.

In [5]:
print(Agente.__doc__)
print()
from agente import INSTRUCCIONES, MODELO, MAX_VUELTAS
print(f"Modelo: {MODELO}   |   Cota de vueltas por consulta: {MAX_VUELTAS}")
print()
print(INSTRUCCIONES)

Conversacion con memoria y acceso a las herramientas del sistema.

Modelo: claude-sonnet-5   |   Cota de vueltas por consulta: 8

Eres el asistente de un taller de mantenimiento de camiones pesados. Ayudas al
jefe de taller a decidir a que vehiculos enviar un mecanico para revisar el
sistema de aire comprimido (APS).

COMO TRABAJAS

Las decisiones no las tomas tu. Las produce un modelo entrenado sobre datos
historicos de la flota, y tu funcion es consultarlo, explicar sus resultados y
traducirlos a terminos que el taller pueda usar.

REGLA PRINCIPAL: toda cifra que menciones debe provenir de una herramienta.
Nunca estimes, aproximes, redondees de memoria ni calcules por tu cuenta un
numero que una herramienta puede darte. Si necesitas un dato, pide la
herramienta correspondiente. Si aun asi no esta disponible, dilo con claridad en
lugar de ofrecer una aproximacion.

Si una herramienta devuelve un error o indica que no encontro algo, comunicalo
tal cual. No inventes un resultado plausib

In [6]:
# Requiere ANTHROPIC_API_KEY en el archivo .env de la raiz del proyecto
asistente = Agente(verboso=True)
print("Agente listo.")

Agente listo.


## 4. Demostracion

Cinco consultas que cubren el uso previsto. El indicador `[herramienta]` muestra
que se ejecuto en cada caso.

### 4.1 La pregunta de la manana

In [7]:
print(asistente.preguntar("Buenos dias. Cuantos camiones tengo que revisar hoy?"))

  [herramienta] resumen_flota({})
Buenos días. Según el sistema, de los **16.000 camiones** evaluados, hoy corresponde revisar **745** (un 4,7% de la flota). Son los que superan el umbral de riesgo vigente.

Un par de datos para ubicar la decisión:
- Seguir esta recomendación tiene un costo estimado de **10.840** (en las unidades de costo del sistema), muy por debajo de no inspeccionar a nadie (**187.500**) o de inspeccionar a toda la flota (**156.250**).
- Eso implica una reducción del **94,2%** frente al escenario de no revisar y esperar a que fallen en ruta.

¿Querés que te pase el listado priorizado si el taller no tiene capacidad para atender los 745 hoy?


### 4.2 Capacidad limitada

El caso realista: el sistema senala mas camiones de los que el taller puede
atender.

In [8]:
print(asistente.preguntar(
    "Solo tengo tres mecanicos disponibles, alcanzo para unas 200 revisiones. "
    "A cuales le doy prioridad?"))

  [herramienta] priorizar_inspecciones({'capacidad': 200})
Con 3 mecánicos alcanzás para 200 revisiones, pero el sistema señala 745 camiones. Te paso los primeros 25 en orden de prioridad (los de mayor puntaje):

| # | Camión | Puntaje |
|---|--------|---------|
| 1 | T15298 | 0.999999 |
| 2 | T09396 | 0.999999 |
| 3 | T03336 | 0.999998 |
| 4 | T13604 | 0.999998 |
| 5 | T09795 | 0.999998 |
| 6 | T14425 | 0.999997 |
| 7 | T01161 | 0.999996 |
| 8 | T14088 | 0.999996 |
| 9 | T01796 | 0.999996 |
| 10 | T11538 | 0.999995 |
| ... | ... | ... |
| 25 | T02800 | 0.99998 |

(La lista completa tiene 200 camiones; puedo pasarte el resto o filtrar por algún criterio si te sirve.)

**Dos cosas importantes:**

1. **Quedan 545 camiones señalados sin atender.** Todos siguen por encima del umbral de riesgo (0.008), o sea que el sistema los considera merecedores de revisión, pero con esta capacidad no llegan hoy.

2. Vas a notar que casi todos los puntajes están pegados a 0.999998, 0.999999, etc. Eso es 

### 4.3 Un camion concreto, y la orden para el mecanico

In [9]:
# Se toma un camion realmente senalado por el sistema, para que la demostracion
# no dependa de un identificador fijo
senalados = herramientas.priorizar_inspecciones(5)["camiones"]
CAMION = senalados[2]["camion_id"]
print(f"Camion elegido para la demostracion: {CAMION} "
      f"(puntaje {senalados[2]['puntaje']})\n")

asistente.reiniciar()
print(asistente.preguntar(
    f"Que pasa con el camion {CAMION}? Si hay que revisarlo dame la orden de trabajo."))

Camion elegido para la demostracion: T03336 (puntaje 0.999998)

  [herramienta] consultar_camion({'camion_id': 'T03336'})
  [herramienta] generar_orden_trabajo({'camion_id': 'T03336'})
## T03336 — Sí, hay que revisarlo

El sistema lo marca para **INSPECCIONAR**. Es un caso de alta prioridad: ocupa el puesto **3 de 16.000** camiones de la flota, así que es de los más urgentes que tenemos en este momento.

(Nota: el puntaje está saturado en el tope de la escala, por lo que la cifra "veces el umbral" no sirve aquí para comparar contra otros camiones — para eso usamos la posición en la flota, que ya te di.)

### Orden de trabajo

**Sistema a revisar:** Aire comprimido (APS)

**Lecturas que motivan la inspección** (por donde debería empezar el mecánico):
- Sensor **ag** — contribución 3.05
- Sensor **ay** — contribución 1.83
- Sensor **az** — contribución 1.70
- Sensor **aa** — contribución 1.35
- Sensor **bi** — contribución 1.23

Estas son las lecturas que empujan la decisión hacia inspec

### 4.4 La objecion previsible

Todo jefe de taller llega a esta pregunta en cuanto ve las primeras
inspecciones. La respuesta correcta no es defender el sistema sino reencuadrar
el criterio.

In [10]:
asistente.reiniciar()
print(asistente.preguntar(
    "Mis mecanicos se quejan. Dicen que revisan camiones que estan perfectos y "
    "que el sistema falla mas de lo que acierta. Que les digo?"))

  [herramienta] evaluar_desempeno({})
Tienen razón en la observación, pero no en la conclusión. Te paso los números reales del histórico para que se los muestres:

- De 745 inspecciones que recomendó el sistema, **384 no encontraron nada** (más de la mitad). Eso confirma la queja: sí se revisan camiones que están perfectos, y va a seguir pasando.
- Pero de las 375 averías reales en la flota, el sistema **detectó 361** y solo se le escaparon 14.
- Con esa política el costo total fue **10.840** (en las unidades de costo del sistema).

Por qué vale la pena aunque "falle" tanto en aciertos puntuales: una avería en ruta cuesta 50 veces lo que cuesta una inspección de más. Así que aunque parezca ineficiente revisar 2.1 camiones por cada avería real detectada, es la forma más barata de operar la flota. Si el sistema fuera más "exigente" para no mandar tantos falsos positivos, dejaría escapar más averías, y cada una de esas es carísima comparada con el rato que se pierde revisando un camión sa

### 4.5 Que tan bien funciono

In [11]:
asistente.reiniciar()
print(asistente.preguntar(
    "Necesito llevarle numeros a la gerencia. Que resultados dio el sistema?"))

  [herramienta] evaluar_desempeno({})
Acá tienes el balance retrospectivo del sistema, para llevar a gerencia:

**Sobre 16.000 camiones evaluados, con 375 averías reales registradas:**

- El sistema recomendó **745 inspecciones**.
- De esas, **361 averías fueron detectadas** a tiempo (de las 375 reales).
- Quedaron **14 averías sin detectar**.
- Hubo **384 inspecciones que no encontraron nada** (innecesarias).
- **Costo total de esta política: 10.840** (en las unidades de costo del sistema).

**Cómo interpretarlo para gerencia:**

- Por cada avería real detectada, se hicieron **2,1 inspecciones** en promedio. Es decir, más de la mitad de las revisiones no encuentran nada — y es esperable. Una avería en ruta cuesta 50 veces más que una revisión que no encuentra nada, así que el sistema prefiere pecar de cauteloso.
- Lo que importa no es "cuántas veces acertó" sino el **costo total**: 10.840 frente a las alternativas extremas (no inspeccionar a nadie, o inspeccionar a todos), que suelen 

## 5. Verificacion: el agente no inventa cifras

Un asistente que responde con soltura a todo es mas peligroso que uno que
reconoce sus limites, porque no hay forma de distinguir sus errores de sus
aciertos. Las tres consultas siguientes buscan justamente que falle.

### 5.1 Un camion que no existe

La respuesta correcta es decir que no aparece. La incorrecta es suponer que el
usuario se equivoco al escribir y responder sobre otro camion.

In [12]:
asistente.reiniciar()
print(asistente.preguntar("Dame el estado del camion T99999."))

  [herramienta] consultar_camion({'camion_id': 'T99999'})
No encuentro el camión **T99999** en la flota evaluada. El sistema tiene registrados identificadores desde **T00000 hasta T15999**, así que revisa si el número se transcribió bien y lo vuelvo a consultar.


### 5.2 Un dato que el sistema no tiene

Ninguna herramienta conoce el kilometraje ni el modelo de los vehiculos. El
agente deberia decirlo en lugar de improvisar.

In [13]:
asistente.reiniciar()
print(asistente.preguntar(
    "Cual es el kilometraje promedio de los camiones que hay que inspeccionar?"))

No dispongo de esa información. El sistema que consulto no maneja datos de kilometraje de los camiones: trabaja con puntajes de riesgo calculados a partir de lecturas de sensores del sistema de aire comprimido (APS), no con variables como kilometraje, antigüedad o uso.

Puedo darte, en cambio:

- El listado de camiones que el sistema recomienda inspeccionar y su puntaje de riesgo (resumen de flota).
- El detalle de sensores que motivan la inspección de un camión puntual.
- Priorización si el taller tiene capacidad limitada.

¿Quieres que te muestre el resumen general de la flota o los camiones priorizados?


### 5.3 Una pregunta que confunde priorizacion con diagnostico

El sistema senala camiones; no dice que componente esta averiado. Y el puntaje
no es una probabilidad. Ambas cosas deberian aclararse.

In [14]:
asistente.reiniciar()
print(asistente.preguntar(
    f"El {CAMION} tiene un 90% de probabilidad de falla, verdad? Que pieza le cambio?"))

  [herramienta] consultar_camion({'camion_id': 'T03336'})
Dos correcciones importantes:

**No es un 90% de probabilidad de falla.** El puntaje del T03336 (0.999998) no es una probabilidad, es un valor de riesgo que se compara contra un umbral para decidir si conviene inspeccionar o no. En este caso el puntaje está **saturado** (en el tope de la escala), así que tampoco sirve compararlo por "veces el umbral" con otros camiones saturados. Lo que sí indica su urgencia real es que está en el **puesto 3 de 16.000** camiones de la flota — es decir, de los más prioritarios que hay.

**El sistema no dice qué pieza cambiar.** Solo prioriza cuáles camiones revisar y qué lecturas motivaron el aviso; el diagnóstico de qué componente falla lo determina el mecánico en la inspección.

Las lecturas que motivan enviar a este camión a revisión son (sensores anonimizados, sin interpretación física conocida):

- **ag** (la que más pesa)
- **ay**
- **az**
- **aa**
- **bi**

La recomendación es: mandar el c

### 5.4 Trazabilidad

Toda cifra que el agente comunico salio de alguna de estas llamadas. El registro
permite auditar la conversacion despues del hecho.

In [15]:
from collections import Counter

conteo = Counter(asistente.herramientas_usadas())
print("Llamadas realizadas en esta sesion:")
for nombre, veces in conteo.most_common():
    print(f"  {nombre:<26} {veces}")
print(f"\nTotal: {sum(conteo.values())}")

Llamadas realizadas en esta sesion:
  consultar_camion           3
  evaluar_desempeno          2
  resumen_flota              1
  priorizar_inspecciones     1
  generar_orden_trabajo      1

Total: 8


## 6. Limitaciones

**El agente hereda todos los limites del modelo.** Si el sistema de priorizacion
esta senalando camiones por desgaste general en lugar de por deterioro del APS
—una posibilidad que el cuaderno de modelado deja abierta—, el agente lo
comunicara con la misma naturalidad con la que comunica cualquier otra cosa. No
valida el sistema, lo hace accesible.

**No actualiza nada.** Trabaja sobre una flota puntuada en un momento dado. Un
uso real exigiria puntuar los camiones al ingresar al taller, lo que requiere
conectar el modelo al flujo de datos de la operacion.

**No sustituye el criterio del jefe de taller.** Un camion puede tener puntaje
bajo y aun asi merecer revision por razones que el sistema no conoce: una queja
del conductor, una reparacion reciente, una ruta particularmente exigente. El
agente aporta un criterio mas, no el unico.

**La verificacion de la Seccion 5 es una muestra, no una garantia.** Comprueba
el comportamiento ante tres formas de pregunta problematica. Un despliegue real
necesitaria un conjunto de casos de prueba mucho mas amplio, ejecutado de forma
automatica ante cada cambio en las instrucciones o en las herramientas.

---

## 7. Como usarlo fuera del cuaderno

    python src/agente.py                                  # conversacion interactiva
    python src/agente.py "cuantos camiones reviso hoy"    # consulta suelta

Requiere `ANTHROPIC_API_KEY` en el archivo `.env` de la raiz del proyecto, y que
los artefactos de `02_modelado.ipynb` esten presentes en `models/`.